# ESA Soil Moisture Data Analysis - Africa

This notebook demonstrates how to download, process, and analyze soil moisture data from ESA CCI using the CDS API.

**Dataset**: Satellite Soil Moisture  
**Source**: Copernicus Climate Data Store  
**Region**: Africa  
**Variables**: SSM (Surface Soil Moisture), RZSM (Root-Zone Soil Moisture)

## 1. Setup and Imports

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import workflow modules
from esa_soil_moisture_workflow import SoilMoistureDownloader, SoilMoistureProcessor

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Create directories
directories = ['data/raw', 'data/processed', 'data/output', 'figures']
for directory in directories:
    Path(directory).mkdir(parents=True, exist_ok=True)

print("Directories created successfully!")

## 2. Configuration

In [ ]:
# Define parameters
YEAR = 2023
MONTH = 6  # June
PRODUCT_TYPE = 'combined'
VARIABLE = 'volumetric_surface_soil_moisture'
TEMPORAL_RES = 'monthly'

# Africa bounding box
AFRICA_BBOX = {
    'north': 40,
    'south': -35,
    'east': 55,
    'west': -20
}

# Sub-regions for detailed analysis
SUBREGIONS = {
    'East Africa': {'north': 15, 'south': -12, 'east': 52, 'west': 28},
    'West Africa': {'north': 25, 'south': 0, 'east': 20, 'west': -20},
    'Southern Africa': {'north': -5, 'south': -35, 'east': 55, 'west': 10},
    'North Africa': {'north': 40, 'south': 15, 'east': 55, 'west': -20}
}

print(f"Configuration set for {YEAR}-{MONTH:02d}")
print(f"Product: {PRODUCT_TYPE}")
print(f"Variable: {VARIABLE}")

## 3. Download Data

**Note**: Uncomment the download code when ready. Downloads can take several minutes.

In [ ]:
# Initialize downloader
downloader = SoilMoistureDownloader(output_dir="./data/raw")

# Download single month
# downloaded_file = downloader.download_soil_moisture(
#     year=YEAR,
#     month=MONTH,
#     product_type=PRODUCT_TYPE,
#     variable=VARIABLE,
#     temporal_resolution=TEMPORAL_RES,
#     bbox=AFRICA_BBOX
# )
# print(f"Downloaded: {downloaded_file}")

print("Download section ready (uncomment to execute)")

In [ ]:
# Download multiple months
# downloaded_files = downloader.download_multiple_months(
#     start_date="2023-01",
#     end_date="2023-06",
#     product_type=PRODUCT_TYPE,
#     variable=VARIABLE,
#     temporal_resolution=TEMPORAL_RES,
#     bbox=AFRICA_BBOX
# )
# print(f"Downloaded {len(downloaded_files)} files")

## 4. Load and Explore Data

In [ ]:
# List available NetCDF files
nc_files = list(Path("./data/raw").glob("*.nc"))
print(f"Found {len(nc_files)} NetCDF files:")
for f in nc_files[:5]:  # Show first 5
    print(f"  - {f.name}")

In [ ]:
# Load a sample file (replace with your actual file)
# sample_file = nc_files[0]
# ds = xr.open_dataset(sample_file)
# 
# # Display dataset information
# print("Dataset Overview:")
# print("=" * 60)
# print(ds)
# print("\nData Variables:")
# print(list(ds.data_vars))
# print("\nDimensions:")
# print(dict(ds.dims))
# print("\nCoordinates:")
# print(list(ds.coords))

## 5. Data Processing

In [ ]:
# Initialize processor
processor = SoilMoistureProcessor(
    input_dir="./data/raw",
    output_dir="./data/processed"
)

# Process a file
# processed_ds = processor.process_workflow(
#     nc_file=sample_file,
#     variables=['sm'],  # Adjust based on actual variable names
#     bbox=AFRICA_BBOX,
#     calculate_stats=True,
#     export_csv=True,
#     export_geotiff=False
# )

## 6. Visualization

### 6.1 Spatial Map of Soil Moisture

In [ ]:
def plot_soil_moisture_map(ds, variable='sm', time_idx=0, title=None, save_path=None):
    """
    Plot soil moisture spatial map
    """
    # Select time slice
    if 'time' in ds[variable].dims:
        data = ds[variable].isel(time=time_idx)
    else:
        data = ds[variable]
    
    # Create figure
    fig = plt.figure(figsize=(14, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())
    
    # Plot data
    im = data.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='YlGnBu',
        vmin=0,
        vmax=0.5,
        cbar_kwargs={'label': 'Soil Moisture (m³/m³)', 'shrink': 0.8}
    )
    
    # Add map features
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.LAND, alpha=0.1)
    
    # Set extent to Africa
    ax.set_extent([-20, 55, -35, 40], crs=ccrs.PlateCarree())
    
    # Add gridlines
    gl = ax.gridlines(draw_labels=True, alpha=0.3)
    gl.top_labels = False
    gl.right_labels = False
    
    # Title
    if title:
        plt.title(title, fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()

# Example usage (uncomment when data is loaded)
# plot_soil_moisture_map(
#     ds,
#     variable='sm',
#     title=f'Surface Soil Moisture - Africa ({YEAR}-{MONTH:02d})',
#     save_path='./figures/soil_moisture_africa.png'
# )

### 6.2 Time Series Analysis

In [ ]:
def plot_time_series(ds, variable='sm', lat=0, lon=20, title=None):
    """
    Plot time series for a specific location
    """
    # Extract time series
    ts = ds[variable].sel(lat=lat, lon=lon, method='nearest')
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot
    ts.plot(ax=ax, linewidth=2, marker='o')
    
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Soil Moisture (m³/m³)', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold')
    else:
        ax.set_title(f'Soil Moisture Time Series (Lat: {lat}°, Lon: {lon}°)', 
                    fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Example usage
# plot_time_series(ds, variable='sm', lat=0, lon=35)

### 6.3 Regional Statistics

In [ ]:
def calculate_regional_stats(ds, variable='sm', subregions=None):
    """
    Calculate statistics for different regions
    """
    if subregions is None:
        subregions = SUBREGIONS
    
    results = []
    
    for region_name, bbox in subregions.items():
        # Subset data
        lat_name = 'lat' if 'lat' in ds.dims else 'latitude'
        lon_name = 'lon' if 'lon' in ds.dims else 'longitude'
        
        subset = ds[variable].sel(
            {lat_name: slice(bbox['north'], bbox['south']),
             lon_name: slice(bbox['west'], bbox['east'])}
        )
        
        # Calculate statistics
        stats = {
            'Region': region_name,
            'Mean': float(subset.mean()),
            'Std': float(subset.std()),
            'Min': float(subset.min()),
            'Max': float(subset.max()),
            'Median': float(subset.median())
        }
        results.append(stats)
    
    return pd.DataFrame(results)

# Example usage
# stats_df = calculate_regional_stats(ds, variable='sm')
# print(stats_df.round(3))

### 6.4 Drought Analysis

In [ ]:
def identify_drought_areas(ds, variable='sm', threshold=0.15, time_idx=0):
    """
    Identify areas with soil moisture below drought threshold
    """
    # Select time slice
    if 'time' in ds[variable].dims:
        data = ds[variable].isel(time=time_idx)
    else:
        data = ds[variable]
    
    # Create drought mask
    drought_mask = data < threshold
    
    # Calculate percentage of area in drought
    drought_percent = (drought_mask.sum() / drought_mask.size) * 100
    
    print(f"Drought Analysis (threshold: {threshold} m³/m³)")
    print(f"Percentage of area in drought: {drought_percent:.2f}%")
    
    # Create visualization
    fig = plt.figure(figsize=(14, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())
    
    # Plot drought areas
    drought_mask.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='RdYlGn_r',
        cbar_kwargs={'label': 'Drought Condition (1=Yes, 0=No)', 'shrink': 0.8}
    )
    
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.set_extent([-20, 55, -35, 40], crs=ccrs.PlateCarree())
    
    gl = ax.gridlines(draw_labels=True, alpha=0.3)
    gl.top_labels = False
    gl.right_labels = False
    
    plt.title(f'Drought Areas - Africa (SM < {threshold} m³/m³)', 
             fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    return drought_mask

# Example usage
# drought_areas = identify_drought_areas(ds, variable='sm', threshold=0.15)

## 7. Export Results

In [ ]:
# Export statistics to CSV
# stats_df.to_csv('./data/output/regional_statistics.csv', index=False)
# print("Statistics exported to CSV")

# Export processed data to NetCDF
# processed_ds.to_netcdf('./data/processed/soil_moisture_processed.nc')
# print("Processed data saved to NetCDF")

## 8. Advanced Analysis Examples

### 8.1 Anomaly Calculation

In [ ]:
def calculate_anomaly(ds, variable='sm', baseline_start='2000-01', baseline_end='2020-12'):
    """
    Calculate soil moisture anomaly relative to climatology
    """
    # Calculate climatology (mean over baseline period)
    climatology = ds[variable].sel(time=slice(baseline_start, baseline_end)).mean(dim='time')
    
    # Calculate anomaly
    anomaly = ds[variable] - climatology
    
    return anomaly

# Example usage
# anomaly = calculate_anomaly(ds, variable='sm')

### 8.2 Correlation with Precipitation

You can combine this with GPM precipitation data for correlation analysis.

In [ ]:
# Placeholder for correlation analysis with GPM data
# This would require loading GPM precipitation data
# and performing spatial correlation analysis

## Summary

This notebook provides a complete workflow for:
1. ✓ Downloading ESA soil moisture data via CDS API
2. ✓ Loading and exploring NetCDF files
3. ✓ Processing and quality control
4. ✓ Spatial and temporal visualization
5. ✓ Regional statistics and drought analysis
6. ✓ Anomaly calculations

**Next Steps:**
- Uncomment download sections to get actual data
- Customize regions of interest
- Integrate with other datasets (GPM, NDVI, etc.)
- Develop specific applications (drought monitoring, agriculture, etc.)